In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [6]:
import pandas as pd

train = pd.read_csv("train.csv")

freq = train["answer"].value_counts().sort_index()

print(freq)

print("\nMost frequent:", freq.max())
print("Least frequent:", freq.min())
print("Required Answer:", freq.max() + freq.min())

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most frequent: 490
Least frequent: 324
Required Answer: 814


In [7]:
import pandas as pd
import string

# Load dataset
train = pd.read_csv("train.csv")

# Lowercase + remove punctuation
translator = str.maketrans("", "", string.punctuation)

clean_prompts = (
    train["prompt"]
    .str.lower()
    .str.translate(translator)
)

# Build vocabulary
vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


In [8]:
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load data
train = pd.read_csv("train.csv")

# Get Prompt for Row ID = 1
prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# Lowercase and remove punctuation
translator = str.maketrans("", "", string.punctuation)
clean_prompt = prompt.lower().translate(translator)

# Split into words
words = clean_prompt.split()

# Remove stop words
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print("Filtered words:", filtered_words)
print("Count:", len(filtered_words))

Filtered words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Count: 13


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
train = pd.read_csv("train.csv")

# Combine prompt and all options
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

# Fit default TF-IDF vectorizer with English stopwords
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(combined_text)

# Vocabulary size
print("Feature columns:", len(vectorizer.get_feature_names_out()))

Feature columns: 2762


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
train = pd.read_csv("train.csv")

# Fit vectorizer on combined text (same as Q3)
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

# Get Row ID = 1
row = train[train["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
optionA_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(prompt_vec, optionA_vec)[0, 0]

print(round(similarity, 4))

0.272


In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
train = pd.read_csv("train.csv")

# Fit TF-IDF on combined text
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0, 0]
        similarities.append(sim)

    predicted = ["A", "B", "C", "D", "E"][similarities.index(max(similarities))]

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / len(train)) * 100

print(f"Percentage: {accuracy:.2f}%")

Percentage: 13.55%


In [13]:
import pandas as pd

# Load training data
train = pd.read_csv("train.csv")

# Frequency of answers
freq = train["answer"].value_counts()

print(freq)

# Top 3 most frequent answers
top3 = freq.index[:3].tolist()
print("Top 3:", top3)

# Compute MAP@3
score = 0

for ans in train["answer"]:
    if ans == top3[0]:
        score += 1.0
    elif ans == top3[1]:
        score += 0.5
    elif ans == top3[2]:
        score += 1/3

map3 = score / len(train)

print("MAP@3 =", round(map3, 6))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3: ['B', 'C', 'A']
MAP@3 = 0.42125


In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
train = pd.read_csv("train.csv")

# Fit TF-IDF on combined prompt + all options
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

total_map3 = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0, 0]
        similarities.append((option, sim))

    # Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    predictions = [x[0] for x in similarities[:3]]

    if row["answer"] == predictions[0]:
        total_map3 += 1.0
    elif row["answer"] == predictions[1]:
        total_map3 += 0.5
    elif row["answer"] == predictions[2]:
        total_map3 += 1/3

map3 = total_map3 / len(train)

print("Final MAP@3:", round(map3, 6))

Final MAP@3: 0.296167
